In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)


In [ ]:
# Reading the saved data pickle file
df  = pd.DataFrame()
df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/data/pkl/sentiment.pkl', compression=None)

In [ ]:
df_sentiment

In [ ]:
df

In [ ]:
df.join(df_sentiment)

In [ ]:
#df_vix.reset_index(level=0, inplace=True)
df_vix[["VIX"]] = ["adj close"].apply(np.int)
#df_vix[["VIX" "articles"]] = df_vix["adj close"].apply(np.int)


In [ ]:
# selecting the VIX and articles
df_vix = df_vix[['VIX', 'articles']]

In [ ]:
df = df_vix[['VIX']].copy()
df


In [ ]:
df_vix = df_vix.reindex(np.random.permutation(df_vix.index))  
df_vix = df_vix[['VIX', 'articles']]

In [ ]:
# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
df

In [ ]:
#import nltk
#nltk.download('vader_lexicon')

In [ ]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
#print(df_vix['articles'])
countNegWord=0
countNeuWord=0
countPosWord=0
for date, row in df_vix.T.iteritems():
    print("The date is:"+str(date))
    try:
        sentence = unicodedata.normalize('NFKD', df_vix.loc[date, 'articles'])#.encode('ascii','ignore').decode('ascii')
        #sentence = unicodedata.normalize('NFKD', df_vix.loc[date, 'articles']).encode('ascii','ignore')
        # add my own sentence Options sentiment.
        #print("#1")
        #print (df_vix.loc[date, 'articles'])
        ss = sid.polarity_scores(sentence)
        #print(sentence)
        #print("#2")
        df.at[date, 'compound'] = ss['compound']
        df.at[date, 'neg'] = ss['neg']
        df.at[date, 'neu'] = ss['neu']
        df.at[date, 'pos'] = ss['pos']
        if ((ss['pos'] + ss['neg']) != 0):
            number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
            if (number != ''):
                df.at[date, 'sentiment'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
                df.at[date, 'sentiment_reflection'] = 100 - 100*(ss['pos'] / (ss['pos'] + ss['neg']))

        countNeg+=ss['neg']
        countNeu+=ss['neu']
        countPos+=ss['pos']
        
        print("============sentiment status============")
        print("compound="+str(df.at[date, 'compound']))
        print("neg="+str(df.at[date, 'neg']))
        print("neu="+str(df.at[date, 'neu']))
        print("pos="+str(df.at[date, 'pos']))
        print("sentiment="+str(df.at[date, 'sentiment']))
        #print("Sentence="+sentence)
        words = sentence.split()
        print("----------start sentenct-------")

        for word in words:
            try:
                #print("word="+word)
                ww = sid.polarity_scores(word)
                if (ww['neg']==1):
                    countNegWord+=1
                elif (ww['neu']==1):
                    countNeuWord+=1
                elif (ww['pos']==1):
                    countPosWord+=1
                else :
                    print("Error ww="+ww)
            except:
                print("Exception for word="+word)

        print("----------end sentenct-------")

        #print("passed")
    except TypeError:
        #print("sentence is")
        #print(sentence)
        #print(df_vix.loc[date, 'articles'])
        print("ERROR")
        df.at[date, 'compound'] = 0
        df.at[date, 'neg'] = 0
        df.at[date, 'neu'] = 0
        df.at[date, 'pos'] = 0



In [ ]:
df["log_VIX"]=np.log(df["VIX"].shift(1)/df["VIX"])

In [ ]:
df["log_VIX"].plot()

In [ ]:
# 2. Foreign volatility indices
corr = df.loc[:,['VIX', 'sentiment', 'sentiment_reflection']].corr()

# heatmap
mask = np.zeros_like(corr)
for i in range(mask.shape[0]):
    mask[i,i]=True
sns.heatmap(corr,annot=True,mask=mask,cmap='coolwarm',vmin=-1,vmax=1)
plt.show()

In [ ]:
sentence = 'The shooting in Paris killed two incon'
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
sid
ss = sid.polarity_scores(sentence)
ss

In [ ]:
df[["sentiment"]]

In [ ]:
for date, sentiment in df.T.iteritems():
    print(date)
    # if (df[date, sentiment]==''):
    #     print("fdsafsda")

In [ ]:
#df.diff().hist(color="k", alpha=0.5, bins=50);
names = ['negative', 'neutral', 'positive']
values = [countNegWord, countNeuWord, countPosWord]

plt.figure(figsize=(9, 3))

#plt.subplot(131)
plt.bar(names, values)

In [ ]:
print(countNegWord)
print(countNeuWord)
print(countPosWord)
# function to add value labels
# function to add value labels
def addlabels(x,y):
    for i in range(len(x)):
        plt.text(i,1,y[i])

names = ['negative','positive']
values = [countNegWord, countPosWord]
#addlabels(names, values)
#plt.text(1, values[1], values[1], ha = 'center')
plt.figure()#figsize=(8, 3),
plt.xlabel("Sentiment Type")
plt.ylabel("Number of words")
plt.bar(names, values)

In [ ]:
#df
df[['pos']]#, 'neg', 'neu']]
# Creating plot
plt.boxplot(df[['pos']])
 
# # show plot
plt.show()

In [ ]:
# df.to_csv(r'/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/data/stata/sentiment.csv', index = False)
df.to_csv(r'/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/data/stata/sentiment.csv', index = False)


In [ ]:
df["sentiment"].dropna(how='all', inplace = True)#values#.to_string().(" ")
#df[['date' 'sentiment']] = pd.DataFrame([ x.split(' ') for x in df['_id'].tolist() ])


In [ ]:
# sentiment_array = df["sentiment"].values
#     for line in df["sentiment"]:
#         line = line.rstrip('\n' + '').split(':')
#         # If line is just empty
#         if line != ['']:
#             print line
# filter(bool, sentiment_array)
# print(sentiment_array)

In [ ]:
#plt.figure();
#df['sentiment'] = df['ForGraph']#.ewm(span=20).mean()
#print(df.values)
#senDF = df.values
#senDF.all.hist(color="k", alpha=0.5, bins=50);

print("Total Neg="+str(countNeg)+" | Total Neu="+str(countNeu)+" | Total Pos="+str(countPos))


fig = df[['VIX']].plot(title='VIX by date').get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/VIX.png")


In [ ]:
df[['sentiment']]

In [ ]:
df[['sentiment']] =df[['sentiment']].apply(pd.to_numeric)
df[['sentiment']].plot()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentiment.png")

df[['sentiment']].ewm(span=20).mean().plot()
#ts=pd.Series(df['sentiment'], index=pd.date_range("25/09/2019", periods=569))
#ts.plot()
# dfplot = df[['sentiment']].plot(title='sentiment by date').apply(pd.to_numeric) 
# fig = dfplot.get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentiment_ewm.png")

# import matplotlib.pyplot as plt

# fig = df[['VIX','sentiment']].plot(title='sentiment by date', secondary_y=True).get_figure()
# dfplot.set_ylim(50, 100)

#sns.boxplot(x='diagnosis', y='area_mean', data=df)
#df.at[date, 'pos'].plot(title='Sentiment by date')
# ts=pd.Series(df['VIX'])#, index=pd.date_range("25/09/2019", periods=569))
# ts.plot()
# df_vix.diff().hist(color="k", alpha=0.5, bins=50);
# ts.plot()
#predictions_df_list['actual_value_ewm'] = predictions_df_list["actual_value"].ewm(span=20).mean()


In [ ]:
df[['sentiment_reflection']] =df[['sentiment_reflection']].apply(pd.to_numeric)
df[['sentiment_reflection']].plot()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentiment_reflection.png")
df[['sentiment_reflection']].ewm(span=20).mean().plot()
#ts=pd.Series(df['sentiment'], index=pd.date_range("25/09/2019", periods=569))
#ts.plot()
# dfplot = df[['sentiment']].plot(title='sentiment by date').apply(pd.to_numeric) 
# fig = dfplot.get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentiment_reflection_ewm.png")

In [ ]:
import matplotlib.pyplot as plt
#df[['sentiment']] = df[['sentiment']].ewm(span=20)
fig, ax1 = plt.subplots(figsize=(20,15))
ax1.plot(df[['VIX']])
ax1.set_xlabel("Date",fontsize=20)
ax1.set_title("Sentiment and VIX", size=26)
ax1.set_ylabel("VIX", size=20)
ax2 = ax1.twinx()
ax2.plot(df[['sentiment']].ewm(span=20).mean(), color='#f97306')# color='#e50000')
ax2.set_ylim([50, 100])
ax2.set_ylabel("sentiment", size=20)
fig.autofmt_xdate()
fig = ax2.get_figure()

fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentimentVIX.png")

In [ ]:
import matplotlib.pyplot as plt
#df[['sentiment']] = df[['sentiment']].ewm(span=20)
fig, ax1 = plt.subplots(figsize=(20,15))
ax1.plot(df[['VIX']], color='#f97306')
ax1.set_xlabel("Date",fontsize=20)
ax1.set_title("Sentiment Reflection and VIX", size=26)
ax1.set_ylabel("VIX", size=20)
ax3 = ax1.twinx()
ax3.plot(df[['sentiment_reflection']].ewm(span=20).mean())# color='#e50000')
ax3.set_ylim([0, 50])
ax3.set_ylabel("sentiment_reflection", size=20)
fig.autofmt_xdate()
fig = ax3.get_figure()

fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentimentReflectionVIX.png")

In [ ]:
import pandas as pd
#if you get an error after executing the code, try adding below:
pd.core.common.is_list_like = pd.api.types.is_list_like

import pandas_datareader.data as web
import datetime

start = datetime.datetime(2019, 9, 25)
end = datetime.datetime(2021, 6, 24)

SP500 = web.DataReader(['sp500'], 'fred', start, end)

In [ ]:
SP500.plot(title='S&P 500 daily returns')


In [ ]:
import matplotlib.pyplot as plt
#df[['sentiment']] = df[['sentiment']].ewm(span=20)
fig, ax1 = plt.subplots(figsize=(20,15))
ax1.plot(SP500, color='#f97306')
ax1.set_xlabel("Date",fontsize=20)
ax1.set_title("VIX and S&P500", size=26)
ax1.set_ylabel("S&P500", size=20)
ax1.set_ylim([2000, 4500])
ax3 = ax1.twinx()
ax3.plot(df[['VIX']])# color='#e50000')
ax3.set_ylim([0, 100])
ax3.set_ylabel("VIX", size=20)
fig.autofmt_xdate()
fig = ax3.get_figure()

fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/VIXandS&P500.png")

In [ ]:
SP500['daily_return'] = (SP500['sp500'].shift(1)/SP500['sp500']) -1

#Drop all Not a number values using drop method.
SP500.dropna(inplace = True)

In [ ]:
SP500['daily_return'].ewm(span=20).mean().plot(title='S&P 500 daily returns')


In [ ]:
fig, axs = plt.subplots(4,figsize=(20,15))
fig.suptitle('Sentiment and VIX', size=26)
fig.autofmt_xdate()
axs[0].set_ylabel("VIX", size=20)
axs[0].plot(df[['VIX']])
axs[1].set_ylabel("sentiment", size=20)
axs[1].plot(df[['sentiment']].ewm(span=20).mean())
axs[2].set_ylabel("S&P500 daily_return", size=20)
axs[2].plot(SP500['daily_return'].ewm(span=20).mean())
axs[3].set_ylabel("S&P500", size=20)
axs[3].plot(SP500['sp500'])
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentimentAndVIX.png")

In [ ]:
fig, axs = plt.subplots(3,figsize=(20,15))
fig.suptitle('S&P500 and S&P500 daily return', size=26)
fig.autofmt_xdate()
axs[0].set_ylabel("S&P500", size=20)
axs[0].plot(SP500['sp500'])
axs[1].set_ylabel("S&P500 Simple Return", size=20)
axs[1].plot(SP500['daily_return'].ewm(span=20).mean())
SP500['sp500_log']=np.log(SP500['sp500'].shift(1)/SP500['sp500'])

axs[2].set_ylabel("S&P500 Log Return", size=20)
axs[2].plot(SP500['sp500_log'].ewm(span=20).mean())

fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentimentAndVIX.png")

In [ ]:
fig, axs = plt.subplots(3,figsize=(20,15))
fig.suptitle('Sentiment and VIX', size=26)
fig.autofmt_xdate()
axs[0].set_ylabel("VIX", size=20)
axs[0].plot(df[['VIX']])
axs[1].set_ylabel("sentiment", size=20)
axs[1].plot(df[['sentiment']].ewm(span=20).mean())
axs[2].set_ylabel("S&P500 daily return", size=20)
axs[2].plot(SP500['daily_return'].ewm(span=20).mean())
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentimentAndVIX.png")

In [ ]:
fig, axs = plt.subplots(2,figsize=(20,15))
fig.suptitle('Sentiment Reflection and VIX', size=26)
fig.autofmt_xdate()
axs[0].set_ylabel("VIX", size=20)
axs[0].plot(df[['VIX']])
axs[1].set_ylabel("sentiment_reflection", size=20)
axs[1].plot(df[['sentiment_reflection']].ewm(span=20).mean())
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentimentReflectionAndVIX.png")

In [ ]:
df[['VIX', 'compound', 'neg', 'neu', 'pos', 'sentiment']]

In [ ]:
#df_sentiment = df['sentiment'].apply(np.int)

In [ ]:
#df_sentiment

In [ ]:
sentence = 'The shooting in Paris killed two'
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
sid
ss = sid.polarity_scores(sentence)
ss

In [ ]:
# import plotly.express as px
# fig = px.box(df, y="VIX")
# fig.show()
df
#sns.boxplot(y='neu', data=df)




# df_sentiment['sentiment'] = df
# target_dist = sns.factorplot(x='data',data=df_sentiment, kind="count", size=20, aspect=1.5, palette="PuBuGn_d")
#target_dist.plot()
# plt.show();
# target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/target_dist_vix.png')
# target_dist = sns.factorplot(x="sentiment", data=df, kind="count", size=20, aspect=1.5, palette="PuBuGn_d")
# plt.show();
# target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/target_dist_sentiment.png')

In [ ]:
# mu, sigma = 100, 15
# #x = mu + sigma * df['VIX']

# # the histogram of the data
# n, bins, patches = plt.hist(df['VIX'])


# plt.xlabel('Smarts')
# plt.ylabel('Probability')
# plt.title('Histogram of IQ')
# plt.text(100, .025, r'$\mu=100,\ \sigma=15$')
# plt.axis([0, 100, 0, 200])
# plt.grid(True)
# plt.show()

In [ ]:
# fig, axes = plt.subplots(10,2,figsize=(25,75))
# # for i,col in enumerate(df['VIX']):
# sns.regplot(1,'VIX',data=df, ax=axes[i,0]) # regression plot
# sns.kdeplot(df['VIX'],df['sentiment'],shade=True, ax=df[i,1]) # kernel density e

In [ ]:
# # Plot each market variable with VKOSPI

# # Columns to plot 
# plot_columns = ['sentiment','VIX']

# # plot
# fig, axes = plt.subplots(10,2,figsize=(25,75))
# for i,col in enumerate(plot_columns):
#     sns.regplot(col,'VIX',data=df, ax=axes[i,0]) # regression plot
#     sns.kdeplot(df[col],df[['VIX']],shade=True, ax=axes[i,1]) # kernel density estimate plot

In [ ]:
# def cor_reg_kde(x,y,data):
#     # Correlation coefficient
#     #print('correlation coefficient:',corr.loc[x,y])
    
#     # Subplot adjust
#     plt.subplots_adjust(wspace=0.5)
    
#     # Plot (regression and kernel density)
#     fig,(ax1,ax2) = plt.subplots(1,2,figsize=(10,5))
#     sns.regplot(x,y,data=data,ax=ax1)
#     sns.kdeplot(data[[x]],data[y],shade=True,ax=ax2)
    
# cor_reg_kde('pos','VIX',df)
# # cor_reg_kde('Indiv_Put_Netbuying_Quantity','Open_interest',df)
# # cor_reg_kde('Indiv_Put_Netbuying_Quantity','Indiv_Call_Netbuying_Quantity',df)
# # cor_reg_kde('Indiv_Call_Netbuying_Quantity','Open_interest',df)

In [ ]:
# # Checking date for empty strings
# for date, row in df_vix.T.iteritems():
#     if type(df_vix.loc[date, 'articles']).__name__ == 'str':
#         print(date)

In [ ]:
# df

In [ ]:
class TextCounts(BaseEstimator, TransformerMixin):
    
    def count_regex(self, pattern, tweet):
        return len(re.findall(pattern, tweet))
    
    def fit(self, X, y=None, **fit_params):
        # fit method is used when specific operations need to be done on the train data, but not on the test data
        return self
    
    def transform(self, X, **transform_params):
        count_words = X.apply(lambda x: self.count_regex(r'\w+', x)) 
        count_mentions = X.apply(lambda x: self.count_regex(r'@\w+', x))
        count_hashtags = X.apply(lambda x: self.count_regex(r'#\w+', x))
        count_capital_words = X.apply(lambda x: self.count_regex(r'\b[A-Z]{2,}\b', x))
        count_excl_quest_marks = X.apply(lambda x: self.count_regex(r'!|\?', x))
        count_urls = X.apply(lambda x: self.count_regex(r'http.?://[^\s]+[\s]?', x))
        # Moreover, it will result in having more words in the tweet
        
        df = pd.DataFrame({'count_words': count_words
                           , 'count_mentions': count_mentions
                           , 'count_hashtags': count_hashtags
                           , 'count_capital_words': count_capital_words
                           , 'count_excl_quest_marks': count_excl_quest_marks
                           , 'count_urls': count_urls
                          })
        
        return df

In [ ]:
tc = TextCounts()
#df_eda = tc.fit_transform(df_vix.articles)


In [ ]:
def show_dist(df, col):
    print('Descriptive stats for {}'.format(col))
    print('-'*(len(col)+22))
    print(df.groupby('VIX')[col].describe())
    bins = np.arange(df[col].min(), df[col].max() + 1)
    g = sns.FacetGrid(df, col='VIX', size=5, hue='VIX', palette="PuBuGn_d")
    g = g.map(sns.distplot, col, kde=False, norm_hist=True, bins=bins)
    plt.show()

In [ ]:
#df_vixs=df[['VIX']]
df_vixs=df_vix
target_dist = sns.factorplot(x="VIX", data=df_vixs, kind="count", size=10, aspect=1.5, palette="PuBuGn_d")
plt.show();
target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/VIX_Histogram.png')

In [ ]:
#df[['sentiment']])#.add(0.5).round().apply(int).options.display.float_format = '{:,.0f}'.format
#round()
df[['sentiment']] = pd.to_numeric(df['sentiment'])

In [ ]:
df[['sentiment']].round()

In [ ]:
target_dist = sns.factorplot(x="sentiment", data=df[['sentiment']].round(0), kind="count", size=15, aspect=1.5, palette="PuBuGn_d")
plt.show();
target_dist.savefig('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/sentiment_Histogram.png')

In [ ]:
#show_dist(df_vixs, 'VIX')

In [ ]:
df

In [ ]:
train_start_date = '2019-09-25'
train_end_date = '2021-06-24'
test_start_date = '2019-09-25'
test_end_date = '2021-06-24'

train = df.loc[train_start_date : train_end_date]
print(test_start_date)
test = df.loc[test_start_date:test_end_date]
print(test_end_date)


In [ ]:
sentiment_score_list = []
for date, row in train.T.iteritems():
    #sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_train = np.asarray(sentiment_score_list)

In [ ]:
sentiment_score_list = []
for date, row in test.T.iteritems():
    print(date)
    #sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_test = np.asarray(sentiment_score_list)

In [ ]:
y_train = pd.DataFrame(train['VIX'])
y_test = pd.DataFrame(test['VIX'])

In [ ]:
# Plot each market variable with VKOSPI

# Columns to plot 
plot_columns = ['VIX','VIX']

# plot
fig, axes = plt.subplots(10,2,figsize=(25,75))
for i,col in enumerate(plot_columns):
    sns.regplot(col,'VIX',data=df, ax=axes[i,0]) # regression plot
    sns.kdeplot(y_train[col],df['VIX'],shade=True, ax=axes[i,1]) # kernel density estimate plot
  

In [ ]:
def cor_reg_kde(x,y,data):
    # Correlation coefficient
    print('correlation coefficient:',corr.loc[x,y])
    
    # Subplot adjust
    plt.subplots_adjust(wspace=0.5)
    
    # Plot (regression and kernel density)
    fig,(ax1,ax2) = plt.subplots(1,2,figsize=(10,5))
    sns.regplot(x,y,data=data,ax=ax1)
    sns.kdeplot(data[x],data[y],shade=True,ax=ax2)
    
cor_reg_kde('Indiv_Future_Netbuying_Quantity','For_Future_Netbuying_Quantity',train_org)
cor_reg_kde('Indiv_Put_Netbuying_Quantity','Open_interest',train_org)
cor_reg_kde('Indiv_Put_Netbuying_Quantity','Indiv_Call_Netbuying_Quantity',train_org)
cor_reg_kde('Indiv_Call_Netbuying_Quantity','Open_interest',train_org)

In [ ]:
from treeinterpreter import treeinterpreter as ti
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report,confusion_matrix

rf = RandomForestRegressor()
print("####################### 1 ######################")
print(y_train)
print("######################## 2 #####################")
print(numpy_df_train)
print("######################## 3 #####################")
rf.fit(numpy_df_train, y_train)

In [ ]:
print(rf.feature_importances_)

In [ ]:
prediction, bias, contributions = ti.predict(rf, numpy_df_test)

In [ ]:
prediction

In [ ]:
contributions

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
print(test_start_date)

idx = pd.date_range(test_start_date, test_end_date)
print(test_end_date)
predictions_df = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'] )

In [ ]:
predictions_df

In [ ]:
#predictions_df.plot() 
#test['VIX'].plot()

predictions_plot = predictions_df.plot()

fig = y_test.plot(ax = predictions_plot).get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/random_forest_without_smoothing_sp500.png")
#matplotlib.pyplot.savefig("C:\Users\Farid\Documents\Project Options predictions\graphs\random forest without smoothing.png")

In [ ]:
ax = predictions_df.rename(columns={"VIX": "predicted_VIX"}).plot(title='Random Forest predicted VIX 1 year')
ax.set_xlabel("Dates")
ax.set_ylabel("VIX")
fig = y_test.rename(columns={"VIX": "actual_VIX"}).plot(ax = ax).get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/random_forest_without_smoothing2_sp500.png")

# colors = ['332288', '88CCEE', '44AA99', '117733', '999933', 'DDCC77', 'CC6677', '882255', 'AA4499']

In [ ]:
test

In [ ]:
# Increasing the VIX by a constant value so that it represents closing VIX during the testing

In [ ]:
from datetime import datetime, timedelta

temp_date = test_start_date
print(test_start_date)
print(datetime.astimezone)
average_last_5_days_test = 0
total_days = 10
for i in range(total_days):
    average_last_5_days_test += test.loc[temp_date, 'VIX']
    # Converting string to date time
    temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
    # Reducing one day from date time
    difference = temp_date + timedelta(days=1)
    # Converting again date time to string
    temp_date = difference.strftime('%Y-%m-%d')
    #print temp_date
average_last_5_days_test = average_last_5_days_test / total_days
print (average_last_5_days_test)

temp_date = test_start_date
average_upcoming_5_days_predicted = 0
for i in range(total_days):
    average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
    # Converting string to date time
    temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
    # Adding one day from date time
    difference = temp_date + timedelta(days=1)
    # Converting again date time to string
    temp_date = difference.strftime('%Y-%m-%d')
    print(temp_date)
average_upcoming_5_days_predicted = average_upcoming_5_days_predicted / total_days
print (average_upcoming_5_days_predicted)
#average train.loc['2013-12-31', 'VIX'] - advpredictions_df.loc['2014-01-01', 'VIX']
difference_test_predicted_VIX = average_last_5_days_test - average_upcoming_5_days_predicted
print (difference_test_predicted_VIX)

In [ ]:
predictions_df

In [ ]:
# Adding 6177 to all the advpredictions_df VIX values
predictions_df['VIX'] = predictions_df['VIX'] + difference_test_predicted_VIX
predictions_df

In [ ]:
ax = predictions_df.rename(columns={"VIX": "predicted_VIX"}).plot(title='Random Forest predicted VIX 1 year after aligning')
ax.set_xlabel("Dates")
ax.set_ylabel("Options Volatility")
fig = y_test.rename(columns={"VIX": "actual_VIX"}).plot(ax = ax).get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/random_forest_with_aligning_sp500.png")



# Smoothing the time series data

# Applying EWM pandas to smooth the VIX volatility

In [ ]:
predictions_df

In [ ]:
predictions_df['ewm'] = predictions_df["VIX"].ewm(span=60).mean()

In [ ]:
predictions_df

In [ ]:
predictions_df['actual_value'] = test['VIX']
predictions_df['actual_value_ewm'] = predictions_df["actual_value"].ewm(span=60).mean()

In [ ]:
predictions_df

In [ ]:
# Changing column names
predictions_df.columns = ['predicted_VIX', 'average_predicted_VIX', 'actual_VIX', 'average_actual_VIX']

In [ ]:
# Now plotting test predictions after smoothing
predictions_plot = predictions_df.plot(title='Random Forest predicted VIX 1 year after aligning & smoothing')
predictions_plot.set_xlabel("Dates")
predictions_plot.set_ylabel("VIX")
fig = predictions_plot.get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/random_forest_after_smoothing_var_sp500.png")


In [ ]:
# Plotting just predict and actual average curves
predictions_df_average = predictions_df[['average_predicted_VIX', 'average_actual_VIX']]
predictions_plot = predictions_df_average.plot(title='Random Forest 1 years after aligning & smoothing')
predictions_plot.set_xlabel("Dates")
predictions_plot.set_ylabel("VIX")
fig = predictions_plot.get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/images/random_forest_after_smoothing2_mean_sp500.png")

In [ ]:
# Increasing the VIX by a constant value so that it represents closing VIX during the testing

In [ ]:
def offset_value(test_start_date, test, predictions_df):
    temp_date = test_start_date
    average_last_5_days_test = 0
    average_upcoming_5_days_predicted = 0
    total_days = 10
    for i in range(total_days):
        average_last_5_days_test += test.loc[temp_date, 'VIX']
        temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
        difference = temp_date + timedelta(days=1)
        temp_date = difference.strftime('%Y-%m-%d')
    average_last_5_days_test = average_last_5_days_test / total_days

    temp_date = test_start_date
    for i in range(total_days):
        average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
        temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
        difference = temp_date + timedelta(days=1)
        temp_date = difference.strftime('%Y-%m-%d')
    average_upcoming_5_days_predicted = average_upcoming_5_days_predicted / total_days
    difference_test_predicted_VIX = average_last_5_days_test - average_upcoming_5_days_predicted
    return difference_test_predicted_VIX

# LogisticRegression

In [ ]:
from treeinterpreter import treeinterpreter as ti
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from datetime import datetime, timedelta

# average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
# # Converting string to date time
# temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
# # Adding one day from date time
# difference = temp_date + timedelta(days=1)
# # Converting again date time to string
# temp_date = difference.strftime('%Y-%m-%d')
        
# start_year = datetime.strptime(train_start_date, "%Y-%m-%d").date().month

prediction_list = []

# Splitting the training and testing data
train_start_date = '2019-09-25'
train_end_date = '2021-06-24'
test_start_date = '2019-09-25'
test_end_date = '2021-06-24'
train = df.loc[train_start_date : train_end_date]
test = df.loc[test_start_date : test_end_date]

# Calculating the sentiment score
sentiment_score_list = []
for date, row in train.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_train = np.asarray(sentiment_score_list)
sentiment_score_list = []
print("#1")
for date, row in test.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_test = np.asarray(sentiment_score_list)
# Generating models
# Linear regression model
lr = LogisticRegression()
print(sys.getsizeof(numpy_df_train))
print(sys.getsizeof(train['VIX']))

print("####################### 1 ######################")
print(train['VIX'])
print("######################## 2 #####################")
print(numpy_df_train)
print("######################## 3 #####################")
lr.fit(numpy_df_train, train['VIX'])

#   if len(np.sum(train['VIX'])) in [len(train['VIX']),0]:
#       print ("all one class")
#       #do something else
#   else:
#       #OK to proceed
#       lenreg.fit(numpy_df_train, train['VIX'])
#       y_pred = lenreg.predict(numpy_df_train)
#       print (metics.accuracy_score(y_test, y_pred))


print("#6")

prediction = lr.predict(numpy_df_test)
prediction_list.append(prediction)
print (train_start_date + ' ' + train_end_date + ' ' + test_start_date + ' ' + test_end_date)
idx = pd.date_range(test_start_date, test_end_date)
#print year
predictions_df_list = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])

difference_test_predicted_VIX = offset_value(test_start_date, test, predictions_df_list)
# Adding offset to all the advpredictions_df VIX values
predictions_df_list['VIX'] = predictions_df_list['VIX'] + difference_test_predicted_VIX
predictions_df_list

# Smoothing the plot
predictions_df_list['ewm'] = predictions_df_list["VIX"].ewm(span=10).mean()
predictions_df_list['actual_value'] = test['VIX']
predictions_df_list['actual_value_ewm'] = predictions_df_list["actual_value"].ewm(span=10).mean()
# Changing column names
predictions_df_list.columns = ['predicted_VIX', 'average_predicted_VIX', 'actual_VIX', 'average_actual_VIX']
predictions_df_list.plot()
predictions_df_list_average = predictions_df_list[['average_predicted_VIX', 'average_actual_VIX']]
predictions_df_list_average.plot()
print("#7")
#     predictions_df_list.show()


In [ ]:
#TEMP
# Columns to plot 
plot_columns = ['sentiment','VIX']

# plot
fig, axes = plt.subplots(10,2,figsize=(25,75))
for i,col in enumerate(plot_columns):
    sns.regplot(col,'VIX',data=train, ax=axes[i,0]) # regression plot
    sns.kdeplot(train[col],train['VIX'],shade=True, ax=axes[i,1]) # kernel density estimate plot
  

In [ ]:
lr.classes_

In [ ]:
lr.coef_[0]

# RandomForestRegressor

In [ ]:
from treeinterpreter import treeinterpreter as ti
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from datetime import datetime, timedelta

# average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
# # Converting string to date time
# temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
# # Adding one day from date time
# difference = temp_date + timedelta(days=1)
# # Converting again date time to string
# temp_date = difference.strftime('%Y-%m-%d')
        
# start_year = datetime.strptime(train_start_date, "%Y-%m-%d").date().month

prediction_list = []
# Splitting the training and testing data
train_start_date = '2019-09-25'
train_end_date = '2021-06-24'
test_start_date = '2019-09-25'
test_end_date = '2021-06-24'
train = df.loc[train_start_date : train_end_date]
test = df.loc[test_start_date:test_end_date]

# Calculating the sentiment score
sentiment_score_list = []
for date, row in train.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_train = np.asarray(sentiment_score_list)
sentiment_score_list = []
for date, row in test.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_test = np.asarray(sentiment_score_list)
# Generating models
rf = RandomForestRegressor(random_state=0)
rf.fit(numpy_df_train, train['VIX'])
#print rf

prediction, bias, contributions = ti.predict(rf, numpy_df_test)
prediction_list.append(prediction)
#print train_start_date + ' ' + train_end_date + ' ' + test_start_date + ' ' + test_end_date
idx = pd.date_range(test_start_date, test_end_date)
#print year
predictions_df_list = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])

difference_test_predicted_VIX = offset_value(test_start_date, test, predictions_df_list)
# Adding offset to all the advpredictions_df VIX values
predictions_df_list['VIX'] = predictions_df_list['VIX'] + difference_test_predicted_VIX
predictions_df_list

# Smoothing the plot
predictions_df_list['ewm'] = predictions_df_list["VIX"].ewm(span=60).mean()
predictions_df_list['actual_value'] = test['VIX']
predictions_df_list['actual_value_ewm'] = predictions_df_list["actual_value"].ewm(span=10).mean()
# Changing column names
predictions_df_list.columns = ['predicted_VIX', 'average_predicted_VIX', 'actual_VIX', 'average_actual_VIX']
predictions_df_list.plot()
predictions_df_list_average = predictions_df_list[['average_predicted_VIX', 'average_actual_VIX']]
predictions_df_list_average.plot()

#     predictions_df_list.show()


In [ ]:
# from IPython.display import Image  
# dot_data = tree.export_graphviz(rf, out_file=None, 
#                      feature_names=['comp', 'neg', 'neu', 'pos'],  
#                      class_names=iris.target_names,  
#                      filled=True, rounded=True,  
#                      special_characters=True)  
# graph = pydotplus.graph_from_dot_data(dot_data)  
# Image(graph.create_png())

# MLP Classifier

In [ ]:
from sklearn.neural_network import MLPClassifier
from datetime import datetime, timedelta

# average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
# # Converting string to date time
# temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
# # Adding one day from date time
# difference = temp_date + timedelta(days=1)
# # Converting again date time to string
# temp_date = difference.strftime('%Y-%m-%d')
        
# start_year = datetime.strptime(train_start_date, "%Y-%m-%d").date().month

prediction_list = []
# Splitting the training and testing data
train_start_date = '2019-09-25'
train_end_date = '2021-06-24'
test_start_date = '2019-09-25'
test_end_date = '2021-06-24'
train = df.loc[train_start_date : train_end_date]
test = df.loc[test_start_date:test_end_date]

# Calculating the sentiment score
sentiment_score_list = []
for date, row in train.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_train = np.asarray(sentiment_score_list)
sentiment_score_list = []
for date, row in test.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_test = np.asarray(sentiment_score_list)

# Generating models
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 100), activation='relu', 
                        solver='lbfgs', alpha=0.005, learning_rate_init = 0.001, shuffle=False) # span = 20 # best 1
mlpc.fit(numpy_df_train, train['VIX'])   
prediction = mlpc.predict(numpy_df_test)

prediction_list.append(prediction)
#print train_start_date + ' ' + train_end_date + ' ' + test_start_date + ' ' + test_end_date
idx = pd.date_range(test_start_date, test_end_date)
#print year
predictions_df_list = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])

difference_test_predicted_VIX = offset_value(test_start_date, test, predictions_df_list)
# Adding offset to all the advpredictions_df VIX values
predictions_df_list['VIX'] = predictions_df_list['VIX'] + difference_test_predicted_VIX
predictions_df_list

# Smoothing the plot
predictions_df_list['ewm'] = predictions_df_list["VIX"].ewm(span=20).mean()
predictions_df_list['actual_value'] = test['VIX']
predictions_df_list['actual_value_ewm'] = predictions_df_list["actual_value"].ewm(span=20).mean()
# Changing column names
predictions_df_list.columns = ['predicted_VIX', 'average_predicted_VIX', 'actual_VIX', 'average_actual_VIX']
predictions_df_list.plot()
predictions_df_list_average = predictions_df_list[['average_predicted_VIX', 'average_actual_VIX']]
predictions_df_list_average.plot()

#     predictions_df_list.show()
    

In [ ]:

#my addition
# import matplotlib.pyplot as plt
# from sklearn.datasets import make_blobs

# n_samples = 200
# blob_centers = ([1, 1], [3, 4], [1, 3.3], [3.5, 1.8])
# data, labels = make_blobs(n_samples=n_samples, 
#                           centers=blob_centers, 
#                           cluster_std=0.5,
#                           random_state=0)


# colours = ('green', 'orange', "blue", "magenta")
# fig, ax = plt.subplots()

# for n_class in range(len(blob_centers)):
#     ax.scatter(data[labels==n_class][:, 0], 
#                data[labels==n_class][:, 1], 
#                c=colours[n_class], 
#                s=30, 
#                label=str(n_class))

In [ ]:
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 100), activation='tanh', 
                         solver='lbfgs', alpha=0.010, learning_rate_init = 0.001, shuffle=False)
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 100), activation='relu', 
                         solver='lbfgs', alpha=0.010, learning_rate_init = 0.001, shuffle=False) # span = 20
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 100), activation='relu', 
                         solver='lbfgs', alpha=0.005, learning_rate_init = 0.001, shuffle=False) # span = 20 # best 1
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 50), activation='relu', 
                         solver='lbfgs', alpha=0.005, learning_rate_init = 0.001, shuffle=False)
                     

In [ ]:
# checking the performance of training data itself
prediction, bias, contributions = ti.predict(rf, numpy_df_train)
idx = pd.date_range(train_start_date, train_end_date)
predictions_df1 = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])
predictions_df1.plot()
train['VIX'].plot()